# Information Crystallization — Seed Phase on GPT-2

**What this notebook does:** runs Phase 1 (seed) of the Information Crystallization algorithm on GPT-2 small (124M params) for a stratified sample of input sequences. For each sequence, it finds:

1. $j^* = \arg\max_j |g_j|$ where $g = \nabla_\theta \log P_\theta(\hat{x} \mid X)$ — the single most gradient-sensitive parameter
2. $t^* = \arg\max_t \|\partial f_\theta / \partial x_t\|$ — the single most gradient-sensitive input position

**What this notebook does NOT do:** Phases 2–4 (growth, prune, verify) are not implemented here. This is a sanity check on the seed step only.

**Known limitation — weight-tying artifact.** GPT-2 ties `lm_head.weight` to `transformer.wte.weight`. The parameter-seed result is therefore ambiguous between "seed localizes in the input embedding" and "seed localizes in the unembedding projection." A clean $S_L \cup S_{\text{supp}}$ test requires an untied model (Pythia-160M, OPT-125M). This notebook does NOT retrofit a post-hoc 'final-region' classifier to paper over the ambiguity.

**Authorship note:** I (Saqib Nazir Bhat) wrote this notebook with AI assistance for scaffolding and debugging. I ran every cell myself, verified the numbers, and can defend every design choice.

## Setup

In [ ]:
# !pip install -q transformers torch matplotlib pandas

In [ ]:
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import transformers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Deterministic seed. The seed phase is already deterministic given fixed
# weights (no sampling, pure forward+backward) — this is belt-and-suspenders.
torch.manual_seed(0)
np.random.seed(0)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'gpt2'  # 124M params
print(f'Device: {DEVICE}')
print(f'torch {torch.__version__}, transformers {transformers.__version__}')

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()  # no dropout; deterministic forward

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_NAME}, total parameters: {n_params:,}')

# Weight-tying check — relevant for interpreting the j* result
tied = model.transformer.wte.weight.data_ptr() == model.lm_head.weight.data_ptr()
print(f'lm_head is tied to wte: {tied}')

## Sample sequences (stratified by query type)

Hand-curated. Big enough to show a distribution, small enough to run on CPU in a few minutes. $N = 17$ is a feasibility check, not a statistical study.

In [ ]:
SEQUENCES = [
    ('syntactic', 'The quick brown fox jumps over the lazy'),
    ('syntactic', 'She opened the door and walked into the'),
    ('syntactic', 'After the rain stopped, the children ran outside to'),
    ('syntactic', 'He picked up the book and started to'),
    ('syntactic', 'The best way to learn programming is to'),

    ('factual', 'The capital of France is'),
    ('factual', 'The author of Harry Potter is'),
    ('factual', 'The largest planet in our solar system is'),
    ('factual', 'The chemical symbol for gold is'),
    ('factual', 'Albert Einstein developed the theory of'),

    ('negation', 'Despite all the warnings, the captain decided to'),
    ('negation', 'Not everyone agreed, but the team chose to'),
    ('negation', 'Although it was raining, they went for a'),
    ('negation', 'Without further delay, the meeting will'),

    ('long-range', 'The scientists who discovered the new particle yesterday have'),
    ('long-range', 'The man standing next to the blue car with the broken window is'),
    ('long-range', 'The cats in the garden behind the old wooden fence are'),
]

print(f'Total sequences: {len(SEQUENCES)}')
print(pd.Series([t for t, _ in SEQUENCES]).value_counts())

## The seed-phase computation

For a single sequence $X$:

1. Forward pass. Let $\ell = f_\theta(X)$ be the final-position logits.
2. $\hat{x} = \arg\max_v \ell_v$.
3. Compute $\log P_\theta(\hat{x} \mid X) = \text{log\_softmax}(\ell)_{\hat{x}}$.
4. `.backward()` gives $\partial \log P / \partial \theta_i$ for every parameter.
5. $j^* = \arg\max_j |g_j|$.

For $t^*$: take the gradient w.r.t. `inputs_embeds` and L2-norm per position.

**Design choices made and why:**

- **Log-probability, not raw logit.** Standard interpretability target. For the winning class, ranks are similar to raw-logit gradient in practice.
- **L2 norm on embedding gradients.** Standard; L1 is also defensible. I report L1 in the record for transparency.
- **GPT-2 small.** Cheapest model with real residual-stream structure. Findings are suggestive, not conclusive for larger models.
- **Single sequence at a time.** Per-sequence embedding gradients; batching adds complexity without speeding up on N = 17.

In [ ]:
def seed_phase(text: str):
    """Run the seed step on a single input sequence."""
    input_ids = tokenizer.encode(text, return_tensors='pt').to(DEVICE)
    seq_len = input_ids.shape[1]
    tokens_str = [tokenizer.decode([t]) for t in input_ids[0].tolist()]
    
    # We go through inputs_embeds so we can backprop to per-token embeddings.
    embeds = model.transformer.wte(input_ids).detach().clone()
    embeds.requires_grad_(True)
    
    for p in model.parameters():
        if p.grad is not None:
            p.grad.zero_()
    model.zero_grad()
    
    outputs = model(inputs_embeds=embeds)
    logits = outputs.logits[0, -1, :]  # [vocab_size]
    
    x_hat_id = int(logits.argmax().item())
    x_hat_str = tokenizer.decode([x_hat_id])
    
    top2 = torch.topk(logits, 2)
    delta = (top2.values[0] - top2.values[1]).item()  # logit margin
    
    log_probs = F.log_softmax(logits, dim=-1)
    target = log_probs[x_hat_id]
    target.backward()
    
    # Find j*
    best_val, best_name = 0.0, None
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        absmax = p.grad.abs().max().item()
        if absmax > best_val:
            best_val = absmax
            best_name = name
    
    seed_layer = classify_param_layer(best_name)
    
    # Find t*
    per_pos_l2 = embeds.grad[0].norm(p=2, dim=-1)
    per_pos_l1 = embeds.grad[0].norm(p=1, dim=-1)
    t_star = int(per_pos_l2.argmax().item())
    
    return {
        'text': text,
        'seq_length': seq_len,
        'x_hat': x_hat_str,
        'x_hat_token_id': x_hat_id,
        'logit_margin': delta,
        'param_seed_name': best_name,
        'param_seed_layer': seed_layer,
        'param_seed_grad': best_val,
        'input_seed_position': t_star,
        'input_seed_position_from_end': seq_len - 1 - t_star,
        'input_seed_token': tokens_str[t_star],
        'input_seed_grad_l2': per_pos_l2[t_star].item(),
        'input_seed_grad_l1': per_pos_l1[t_star].item(),
    }

def classify_param_layer(name):
    """Map a GPT-2 parameter name to a location class.
    
    IMPORTANT: on GPT-2, `transformer.wte.weight` is the input embedding AND
    is tied to `lm_head.weight` (unembedding). A seed on `wte.weight` is
    therefore AMBIGUOUS — it could be driven by either role. Label reflects this.
    """
    if name is None:
        return 'unknown'
    if 'wte' in name:
        return 'embedding/unembedding (tied on GPT-2)'
    if 'wpe' in name:
        return 'position_embedding'
    if 'lm_head' in name:
        return 'unembedding'  # distinct tensor only on untied models
    if '.h.' in name:
        block_idx = int(name.split('.h.')[1].split('.')[0])
        return f'block_{block_idx}'
    if 'ln_f' in name:
        return 'final_layernorm'
    return 'other'

# Sanity check
result = seed_phase('The capital of France is')
for k, v in result.items():
    print(f'{k:30s}: {v}')

## Run across all sequences

In [ ]:
records = []
for query_type, text in SEQUENCES:
    r = seed_phase(text)
    r['query_type'] = query_type
    records.append(r)
    print(f'[{query_type:10s}] "{text}" -> "{r["x_hat"]}" | seed={r["param_seed_layer"]} | t*={r["input_seed_position"]}/{r["seq_length"]-1} ({r["input_seed_token"]!r})')

df = pd.DataFrame(records)
df

## Finding 1: Where does $j^*$ live?

On weight-tied GPT-2 this finding is ambiguous — see the top-of-notebook caveat. Reporting the raw distribution without a post-hoc 'final-region' classifier.

In [ ]:
layer_counts = df['param_seed_layer'].value_counts()
print('Seed-parameter location distribution:')
print(layer_counts)

N_LAYERS = model.config.n_layer
print(f'\n(GPT-2 small has {N_LAYERS} transformer blocks.)')
print('For a clean S_L vs S_supp test, re-run on a model with untied')
print('embeddings — Pythia-160M or OPT-125M are the closest analogues.')

In [ ]:
# Stratified by query type — still ambiguous under weight-tying, but recorded for transparency
by_type = df.groupby('query_type')['param_seed_layer'].value_counts()
print('Seed location by query type:')
print(by_type)

## Finding 2: Where does $t^*$ live?

No weight-tying confound here. The paper conjectures $T = \{x_{n-k}, \ldots, x_n\} \cup T_{\text{sal}}$; the corrected claim (Conjecture 2 in the paper) is that $T$ is identified by attention weight concentration rather than positional recency.

In [ ]:
df['seed_is_last'] = df['input_seed_position'] == (df['seq_length'] - 1)
df['seed_is_first_3'] = df['input_seed_position'] < 3

print('Input-seed position statistics:')
print(f"  seed = last token:         {df['seed_is_last'].mean():.1%}")
print(f"  seed in first 3 tokens:    {df['seed_is_first_3'].mean():.1%}")
print(f"  mean distance from end:    {df['input_seed_position_from_end'].mean():.2f} tokens")
print()
print('By query type (mean distance from end; n in parentheses):')
counts = df.groupby('query_type').size()
means = df.groupby('query_type')['input_seed_position_from_end'].mean()
for q in means.index:
    print(f"  {q:12s} n={counts[q]:2d}  mean={means[q]:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

layer_counts.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title(f'Seed parameter $j^*$ — location\n(GPT-2 small, {N_LAYERS} blocks; weight-tying caveat)')
axes[0].set_xlabel('Count')

axes[1].hist(df['input_seed_position_from_end'], bins=range(0, df['seq_length'].max()+1),
             color='steelblue', edgecolor='black')
axes[1].set_title('Input seed $t^*$ — distance from end')
axes[1].set_xlabel('n - 1 - t* (0 = last token)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('seed_phase_results.png', dpi=120)
plt.show()

## What this does and does not show

**Shows:**
- The seed step is deterministic, reproducible, and produces interpretable outputs on a real transformer.
- $t^*$ is concentrated away from the suffix for this sample — consistent with the paper's attention-concentration conjecture.

**Does not show:**
- $S_L$ vs $S_{\text{supp}}$ decomposition — weight-tied GPT-2 cannot adjudicate this.
- Full $(S, T)$ minimum — Phases 2–4 unimplemented.
- Statistical significance — $N = 17$ is a feasibility check. The long-range bucket has only $n = 3$.
- Generalization to larger models.

**Next experiment:** Pythia-160M (untied embeddings) to get a clean $S_L$ signal.